# Calculating OCR accuracy with WER and CER

This notebook is the final step in the THISTLE pipeline. It calculates word error rate (WER) and character error rate (CER) for all OCR outputs generated in the previous steps, and produces a summary comparing the performance of each method.

## What are WER and CER?

Both metrics compare a system's output to a manually transcribed 'ground truth'. Word Error Rate (WER) measures the proportion of words that differ between the OCR output and the ground truth. A WER of 0.0 indicates a perfect match; a WER of 1.0 means every word is wrong. Character Error Rate (CER) performs the same calculation as WER, but at the character rather than word level. Generally, CER is more sensitive to small OCR errors (such as a single misread character within a word) than WER.

Both metrics are calculated using the [`jiwer`](https://github.com/jitsi/jiwer) library in Python.

## What is being compared?

This notebook compares the following outputs against the manually transcribed ground truth: `ocr`, the original OCR from ProQuest; `tesseract_ocr`, the OCR extraced by Tesseract after image preprocessing; and the OCR corrected by base and fine-tuned LLMs.

**Note:** remember to double-check the `ocr_cols` list in Step 3 and, if needed, update this variable to match the columns in your dataset.


## For working in Colab

## Connect to the GPU:
In the 'Runtime' menu, click on 'Change runtime type' and select 'T4 GPU' under 'Hardware accelerator'. **NOTE:** you will need to be logged in with your Google account to connect your Drive and to use the GPU. Free access is limited; for larger projects, you may need to consider working on an HPC.

## Connect your Google Drive:
Run the code block below to mount your Google Drive.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


## Step 1: Install and import dependencies
Run this cell first to install all required Python packages.

In Google Colab, packages are installed for your current session and will need reinstalling if you reconnect.

If you are working locally, run this step only once per environment, or install via `requirements.txt` instead (see `code/README.md`).


In [ ]:
!pip install pandas jiwer

import pandas as pd
from jiwer import wer, cer

## Step 2: Import data

This notebook takes the dataframe produced in the previous steps as its input. It should contain a column with the ground truth (manually transcribed text) and one or more columns with OCR or LLM-corrected outputs to compare against it.

Update the path below to point to your combined output CSV.

In [ ]:
# paste the path to the combined post-OCR output CSV from the previous steps
# example (Colab): "/content/drive/My Drive/your_project_folder/post_ocr_output.csv"
# example (local): "/Users/yourname/Documents/THISTLE_project/post_ocr_output.csv"
df = pd.read_csv('insert_your_path_to/post_ocr_output.csv')

## Step 3: Specify columns to compare

Update the `ocr_cols` list to include the names of the columns you want to evaluate. Each column will be compared against the ground truth column (`manual_transcription`). If your ground truth column has a different name, update that too.

In [ ]:
# list the columns to evaluate — update to match your dataset
# options tested in this project: 'ocr', 'tesseract_ocr', 'llama_output', 'gemma2_output', 'mistral_output', 'llama_fine_tuned_output'
ocr_cols = ['ocr', 'tesseract_ocr', 'llama_output', 'gemma2_output', 'mistral_output', 'llama_fine_tuned_output']

# name of the ground truth column
ground_truth_col = 'manual_transcription'  # update if your column has a different name

## Step 4: Calculate WER and CER

This cell calculates WER and CER for each row in each comparison column. Results are added to the dataframe as new columns (e.g. `ocr_wer`, `ocr_cer`).

In [ ]:
for col in ocr_cols:
    if col not in df.columns:
        print(f'Warning: column "{col}" not found in dataframe — skipping.')
        continue
    df[f'{col}_wer'] = df.apply(
        lambda row: wer(str(row[ground_truth_col]), str(row[col]))
        if pd.notna(row[ground_truth_col]) and pd.notna(row[col]) else None,
        axis=1
    )
    df[f'{col}_cer'] = df.apply(
        lambda row: cer(str(row[ground_truth_col]), str(row[col]))
        if pd.notna(row[ground_truth_col]) and pd.notna(row[col]) else None,
        axis=1
    )
print(df.head())

## Step 5: Print a summary of results

This cell prints the mean WER and CER for each method, making it easy to compare performance at a glance.

In [ ]:
print(f'{'Method':<35} {'Mean WER':>10} {'Mean CER':>10}')
print('-' * 57)
for col in ocr_cols:
    if f'{col}_wer' in df.columns:
        mean_wer = df[f'{col}_wer'].mean()
        mean_cer = df[f'{col}_cer'].mean()
        print(f'{col:<35} {mean_wer:>10.4f} {mean_cer:>10.4f}')

## Step 6: Save the dataframe

Update the path below to save the results to your preferred location.

In [ ]:
# paste the path where you would like to save the scored output CSV
# example (Colab): "/content/drive/My Drive/your_project_folder/wer_cer_scores.csv"
# example (local): "/Users/yourname/Documents/THISTLE_project/wer_cer_scores.csv"
df.to_csv('insert_your_output_path/wer_cer_scores.csv', index=False)
print('Saved successfully.')